# 02 - Prueba de reglas de calidad, perfilado y MDM

Carga los datasets sintéticos y ejecuta perfilado, calidad y MDM (sin necesidad de Ollama/Docker).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if (Path.cwd().name == 'notebooks') else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from app.services.profiler import profile_dataframe
from app.services.quality_rules import evaluate_quality
from app.services.mdm import detect_duplicates

syn = ROOT / 'data' / 'synthetic'
tables = {n: pd.read_csv(syn / f'{n}.csv') for n in ['clientes', 'productos', 'ventas', 'proveedores']}
list(tables.keys())

In [ ]:
# Perfilado de clientes
prof = profile_dataframe('clientes', tables['clientes'])
print('Filas:', prof.rows, '| Duplicadas:', prof.duplicate_rows)
print('Sensibles:', prof.possible_sensitive_fields)
print('Duplicados por clave:', prof.duplicates)

In [ ]:
# Calidad (con integridad referencial cruzada)
for name, df in tables.items():
    rep = evaluate_quality(name, df, related=tables)
    print(f'{name}: score={rep.quality_score} riesgo={rep.risk_level} hallazgos={len(rep.findings)}')

In [ ]:
# Hallazgos de ventas (integridad referencial + total mal calculado)
rep = evaluate_quality('ventas', tables['ventas'], related=tables)
pd.DataFrame([f.model_dump() for f in rep.findings])

In [ ]:
# MDM en clientes
res = detect_duplicates('cliente', tables['clientes'])
print('Duplicados detectados:', res.duplicates_detected)
for g in res.duplicate_groups[:5]:
    print(g.match_type, g.confidence, g.member_ids)